In [1]:
from google.colab import drive
drive.mount('/content/drive/')

Mounted at /content/drive/


In [6]:

!pip install h5py pandas rasterio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 93.7 MB/s eta 0:00:00


In [4]:
import h5py
import os
import pandas as pd

# Specify the root directory containing yearly folders with HDF5 files
root_directory = '/content/drive/MyDrive/Wildfire Spread Prevention Project/wildfirespreadts/hdf5'

# Year you want to process (Change this to '2019', '2020', etc. after each run)
year = '2018'
year_directory = os.path.join(root_directory, year)

# List to collect event data
fire_events = []

# Safeguard in case directory is missing
if not os.path.exists(year_directory):
    print(f"Directory not found: {year_directory}")
else:
    file_list = [f for f in os.listdir(year_directory) if f.endswith('.hdf5')]

    for idx, file_name in enumerate(file_list):
        file_path = os.path.join(year_directory, file_name)

        try:
            with h5py.File(file_path, 'r') as f:
                lat_min = f.attrs.get('lat_min', None)
                lat_max = f.attrs.get('lat_max', None)
                lon_min = f.attrs.get('lon_min', None)
                lon_max = f.attrs.get('lon_max', None)

                if None not in (lat_min, lat_max, lon_min, lon_max):
                    fire_events.append({
                        'fire_id': file_name.replace('.hdf5', ''),
                        'year': year,
                        'lat_min': lat_min,
                        'lat_max': lat_max,
                        'lon_min': lon_min,
                        'lon_max': lon_max
                    })

            # Optional: Print progress every 50 files
            if idx % 50 == 0:
                print(f"Processed {idx} files...")

        except Exception as e:
            print(f"Error reading {file_name}: {e}")

# Save the results
output_csv_path = f'/content/fire_event_locations_{year}.csv'
fire_events_df = pd.DataFrame(fire_events)
fire_events_df.to_csv(output_csv_path, index=False)

print(f"Saved {len(fire_events_df)} fire events for {year} to {output_csv_path}")
fire_events_df.head()

Processed 0 files...
Processed 50 files...
Processed 100 files...
Processed 150 files...
Saved 0 fire events for 2018 to /content/fire_event_locations_2018.csv


""


In [7]:
import rasterio
import os
import pandas as pd

# Root directory for extracted .tif data
tif_root_directory = '/content/drive/MyDrive/Wildfire Spread Prevention Project/wildfirespreadts/extracted_data'

# List to collect bounding boxes
bounding_boxes = []

# Loop over years
for year in ['2018', '2019', '2020', '2021']:
    year_dir = os.path.join(tif_root_directory, year)
    if os.path.exists(year_dir):
        file_list = [f for f in os.listdir(year_dir) if f.endswith('.tif')]

        for file_name in file_list:
            file_path = os.path.join(year_dir, file_name)
            fire_id = file_name.replace('.tif', '')

            try:
                with rasterio.open(file_path) as src:
                    bounds = src.bounds  # bounds: left, bottom, right, top
                    bounding_boxes.append({
                        'fire_id': fire_id,
                        'year': year,
                        'lon_min': bounds.left,
                        'lat_min': bounds.bottom,
                        'lon_max': bounds.right,
                        'lat_max': bounds.top
                    })
            except Exception as e:
                print(f"Error reading {file_name}: {e}")

# Save to a DataFrame
bounding_boxes_df = pd.DataFrame(bounding_boxes)
bounding_boxes_df.to_csv('/content/fire_event_locations_from_tif.csv', index=False)

print("Saved bounding boxes extracted from GeoTIFFs!")
bounding_boxes_df.head()

Saved bounding boxes extracted from GeoTIFFs!


""


In [8]:
import os

tif_root_directory = '/content/drive/MyDrive/Wildfire Spread Prevention Project/wildfirespreadts/extracted_data'
for year in ['2018', '2019', '2020', '2021']:
    year_dir = os.path.join(tif_root_directory, year)
    if os.path.exists(year_dir):
        print(f"Year {year} has {len(os.listdir(year_dir))} files.")

Year 2018 has 176 files.
Year 2019 has 74 files.
Year 2020 has 201 files.
Year 2021 has 156 files.


In [10]:
import os
import rasterio

# Go into a fire folder
fire_folder = '/content/drive/MyDrive/Wildfire Spread Prevention Project/wildfirespreadts/extracted_data/2018/fire_21997854'

# List tif files inside that fire
tif_files = [f for f in os.listdir(fire_folder) if f.endswith('.tif')]
print(tif_files)

# Pick the first .tif file
example_tif = os.path.join(fire_folder, tif_files[0])

# Now open it
with rasterio.open(example_tif) as src:
    print("CRS:", src.crs)
    print("Bounds:", src.bounds)
    print("Width, Height:", src.width, src.height)
    print("Transform:", src.transform)

['2018-08-09.tif', '2018-08-10.tif', '2018-08-12.tif', '2018-08-04.tif', '2018-08-05.tif', '2018-08-06.tif', '2018-08-07.tif', '2018-08-08.tif', '2018-08-03.tif', '2018-08-11.tif']
CRS: EPSG:32610
Bounds: BoundingBox(left=724875.0, bottom=3929625.0, right=818625.0, top=4043625.0)
Width, Height: 250 304
Transform: | 375.00, 0.00, 724875.00|
| 0.00,-375.00, 4043625.00|
| 0.00, 0.00, 1.00|


In [11]:
!pip install pyproj rasterio

import os
import rasterio
from pyproj import Transformer
import pandas as pd

# Initialize UTM to WGS84 (lat/lon) transformer
# Your CRS is EPSG:32610 -> UTM Zone 10N
transformer = Transformer.from_crs("epsg:32610", "epsg:4326", always_xy=True)

# Root path
tif_root_directory = '/content/drive/MyDrive/Wildfire Spread Prevention Project/wildfirespreadts/extracted_data'

# Collect results
bounding_boxes = []

# Loop through years
for year in ['2018', '2019', '2020', '2021']:
    year_dir = os.path.join(tif_root_directory, year)
    if os.path.exists(year_dir):
        fire_folders = [f for f in os.listdir(year_dir) if os.path.isdir(os.path.join(year_dir, f))]

        for fire_folder in fire_folders:
            fire_path = os.path.join(year_dir, fire_folder)
            tif_files = [f for f in os.listdir(fire_path) if f.endswith('.tif')]

            if tif_files:
                # Pick first available .tif file
                tif_file_path = os.path.join(fire_path, tif_files[0])

                try:
                    with rasterio.open(tif_file_path) as src:
                        bounds = src.bounds  # UTM bounds: left, bottom, right, top

                        # Convert all four corners to lat/lon
                        lon_min, lat_min = transformer.transform(bounds.left, bounds.bottom)
                        lon_max, lat_max = transformer.transform(bounds.right, bounds.top)

                        bounding_boxes.append({
                            'fire_id': fire_folder,  # fire_xxxxx
                            'year': year,
                            'lat_min': lat_min,
                            'lat_max': lat_max,
                            'lon_min': lon_min,
                            'lon_max': lon_max
                        })
                except Exception as e:
                    print(f"Error reading {fire_folder}: {e}")

# Save to CSV
bounding_boxes_df = pd.DataFrame(bounding_boxes)
bounding_boxes_df.to_csv('/content/fire_event_locations_final.csv', index=False)

print(f"Saved {len(bounding_boxes_df)} bounding boxes from GeoTIFFs!")
bounding_boxes_df.head()

Saved 607 bounding boxes from GeoTIFFs!


,fire_id,year,lat_min,lat_max,lon_min,lon_max
0,fire_21997854,2018,35.484733,36.484967,-120.521207,-119.443503
1,fire_21890058,2018,41.672369,42.662846,-117.056161,-115.846458
2,fire_22141426,2018,45.573348,46.572606,-120.473560,-119.355718
3,fire_21748798,2018,34.174542,35.155756,-112.977778,-111.719708
4,fire_22257911,2018,39.727985,40.733876,-122.651390,-121.624765


In [13]:
import shutil
shutil.move('/content/fire_event_locations_final.csv', '/content/drive/MyDrive/Wildfire Spread Prevention Project/fire_event_locations_final.csv')

'/content/drive/MyDrive/Wildfire Spread Prevention Project/fire_event_locations_final.csv'